In [ ]:
from pathlib import Path
import json

In [ ]:
def write_nested_schema(schema_dict: dict[str, dict], prop: str) -> str:

    schema_section = schema_dict[prop]
    schema_str = f"class {prop}_Base(BaseModel):\n\n"
    for key, val in schema_section["components"].items():
        dtype = val["type"]
        if val["list"] == True:
            dtype = f"list[{dtype}]"

        if val["optional"] == True:
            dtype += " | None"

        field_str = f"    {key+':':<25}{dtype}"

        if val["description"] is not None:
            field_str += f' = Field(description="{val["description"]}")'

        schema_str += field_str+"\n"

    schema_str += "\n\n"

    subclasses = []
    for key, subclass in schema_section.items():
        if key in ["components", "multiple_allowed"]:
            continue

        subclasses.append([key, f"{key}_{prop}"])
        subclass_str = f"class {key}_{prop}({prop}_Base):\n"

        if subclass["additional_components"] is None:
            subclass_str += "    pass\n\n\n"
            schema_str += subclass_str
            continue
        else:
            subclass_str += "\n"
            for subkey, subval in subclass["additional_components"].items():
                dtype = subval["type"]
                if subval["list"] == True:
                    dtype = f"list[{dtype}]"

                if subval["optional"] == True:
                    dtype += " | None"

                field_str = f"    {subkey+':':<25}{dtype}"

                if subval["description"] is not None:
                    field_str += f' = Field(description="{subval["description"]}")'

                subclass_str += field_str + "\n"

        schema_str += subclass_str+"\n\n"

    return schema_str, subclasses


schema_file = Path("/home/brock/Documents/github/orca-prop-parse/orca_spec/orca_prop_spec_py.json").read_text()
schema_dict = json.loads(schema_file)
prop = "Calculation_Timings"
schema_str, subclasses = write_nested_schema(schema_dict, prop)

print("from pydantic import BaseModel, Field\n\n")

print(schema_str)

print(f"class {prop}(BaseModel):\n")
for i in subclasses:
    print(f"    {i[0]+":":<15}{i[1]}")

In [ ]:
#print(json.dumps(schema_dict["Spin_Spin_Coupling"]["components"], indent=2))
prop = "Calculation_Timings"
print(f"class {prop.replace("_", "")}(BaseModel):\n")
for key, val in schema_dict[prop]["components"].items():

    dtype = val["type"]
    if val["list"] == True:
        dtype = f"list[{dtype}]"

    if val["optional"] == True:
        dtype += " | None"

    field_str = f"    {key+':':<12}{dtype}"

    if val["description"] is not None:
        field_str += f' = Field(description="{val["description"]}")'
    print(field_str)